# Feature Engineering – DOGE/USDT Technical Indicators

El objetivo de este notebook es transformar el dataset bruto de mercado en un conjunto de datos enriquecido mediante técnicas de feature engineering orientadas al análisis financiero y al trading algorítmico.

Para ello, se generan indicadores técnicos, métricas de volatilidad, retornos, variables derivadas y etiquetas supervisadas que permitirán entrenar y evaluar modelos predictivos capaces de identificar patrones de comportamiento en el mercado de criptomonedas.

In [1]:
# ============================================================
# 02 - Feature engineering: DOGE/USDT technical indicators
# ============================================================

import os
import numpy as np
import pandas as pd

RAW_PATH = "../data/raw/DOGEUSDT_5m_binance_2017_2026.csv"
OUTPUT_DIR = "../data/processed"
OUTPUT_FILE = "DOGEUSDT_5m_binance_2017_2026_features.csv"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE)

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(RAW_PATH, parse_dates=["open_time", "close_time"])
df = df.sort_values("open_time").reset_index(drop=True)

print("Loaded raw dataset")
print("Shape:", df.shape)
print("Date range:", df["open_time"].min(), "->", df["open_time"].max())

display(df.head())

Loaded raw dataset
Shape: (723409, 11)
Date range: 2019-07-05 12:00:00 -> 2026-05-23 11:10:00


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume
0,2019-07-05 12:00:00,0.004490,0.004600,0.003760,0.004350,281690734.0,2019-07-05 12:04:59.999,1.213789e+06,1841,131946467.0,572788.485474
1,2019-07-05 12:05:00,0.004340,0.004340,0.004000,0.004022,151285521.0,2019-07-05 12:09:59.999,6.310450e+05,963,41885000.0,173783.101331
2,2019-07-05 12:10:00,0.004022,0.004069,0.003800,0.003870,182657540.0,2019-07-05 12:14:59.999,7.079442e+05,1285,58720314.0,228693.805868
3,2019-07-05 12:15:00,0.003861,0.003940,0.003859,0.003929,73986015.0,2019-07-05 12:19:59.999,2.885708e+05,535,43864831.0,171156.394506
4,2019-07-05 12:20:00,0.003929,0.003964,0.003870,0.003877,67030342.0,2019-07-05 12:24:59.999,2.627509e+05,399,27619614.0,108776.180726


In [2]:
# ============================================================
# Technical indicators
# ============================================================

# Returns
df["return_1"] = df["close"].pct_change()
df["log_return_1"] = np.log(df["close"] / df["close"].shift(1))

# Moving averages
df["sma_20"] = df["close"].rolling(window=20).mean()
df["ema_10"] = df["close"].ewm(span=10, adjust=False).mean()
df["ema_50"] = df["close"].ewm(span=50, adjust=False).mean()
df["ema_200"] = df["close"].ewm(span=200, adjust=False).mean()

# EMA ratios
df["ema10_ema50_ratio"] = df["ema_10"] / df["ema_50"]
df["ema50_ema200_ratio"] = df["ema_50"] / df["ema_200"]
df["sma20_ema50_ratio"] = df["sma_20"] / df["ema_50"]

# Rolling volatility: 1 hour = 12 candles of 5 minutes
df["volatility_1h"] = df["log_return_1"].rolling(window=12).std()

# Z-score over 1 hour
rolling_mean_1h = df["close"].rolling(window=12).mean()
rolling_std_1h = df["close"].rolling(window=12).std()
df["zscore_close_1h"] = (df["close"] - rolling_mean_1h) / rolling_std_1h

# RSI 14
delta = df["close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))

# MACD: 12, 26, 9
ema_12 = df["close"].ewm(span=12, adjust=False).mean()
ema_26 = df["close"].ewm(span=26, adjust=False).mean()

df["macd"] = ema_12 - ema_26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
df["macd_hist"] = df["macd"] - df["macd_signal"]

# Bollinger Bands: 20 periods, 2 std
bb_window = 20
bb_std = 2

bb_mid = df["close"].rolling(window=bb_window).mean()
bb_std_dev = df["close"].rolling(window=bb_window).std()

df["bb_mid"] = bb_mid
df["bb_upper"] = bb_mid + bb_std * bb_std_dev
df["bb_lower"] = bb_mid - bb_std * bb_std_dev
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["bb_mid"]
df["bb_percent"] = (df["close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])

# ATR 14
high_low = df["high"] - df["low"]
high_close_prev = (df["high"] - df["close"].shift(1)).abs()
low_close_prev = (df["low"] - df["close"].shift(1)).abs()

true_range = pd.concat([high_low, high_close_prev, low_close_prev], axis=1).max(axis=1)
df["atr_14"] = true_range.rolling(window=14).mean()

print("Features created")
print("Shape:", df.shape)

display(df.tail())

Features created
Shape: (723409, 32)


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,rsi_14,macd,macd_signal,macd_hist,bb_mid,bb_upper,bb_lower,bb_width,bb_percent,atr_14
723404,2026-05-23 10:50:00,0.09957,0.09961,0.09951,0.09952,2019387.0,2026-05-23 10:54:59.999,201036.76574,672,1267731.0,...,52.873563,-0.000053,-0.000090,0.000038,0.099524,0.099679,0.099368,0.003117,0.488716,0.000115
723405,2026-05-23 10:55:00,0.09952,0.09954,0.09950,0.09952,284994.0,2026-05-23 10:59:59.999,28361.11493,286,211374.0,...,50.602410,-0.000051,-0.000082,0.000031,0.099520,0.099673,0.099368,0.003071,0.498364,0.000110
723406,2026-05-23 11:00:00,0.09952,0.09955,0.09946,0.09948,591135.0,2026-05-23 11:04:59.999,58808.86284,561,257920.0,...,51.219512,-0.000053,-0.000077,0.000024,0.099516,0.099669,0.099364,0.003067,0.380429,0.000110
723407,2026-05-23 11:05:00,0.09948,0.09962,0.09947,0.09954,1837323.0,2026-05-23 11:09:59.999,182909.11863,1442,1026953.0,...,58.536585,-0.000049,-0.000071,0.000022,0.099516,0.099668,0.099364,0.003059,0.578827,0.000114
723408,2026-05-23 11:10:00,0.09954,0.09961,0.09954,0.09956,286859.0,2026-05-23 11:14:59.999,28558.43106,459,182916.0,...,58.536585,-0.000043,-0.000065,0.000022,0.099515,0.099666,0.099364,0.003029,0.649286,0.000111


In [3]:
# ============================================================
# Supervised learning labels
# ============================================================

horizons = [1, 3, 6, 12]

for h in horizons:
    df[f"future_close_{h}"] = df["close"].shift(-h)
    df[f"return_{h}"] = df[f"future_close_{h}"] / df["close"] - 1
    df[f"up_{h}"] = (df[f"return_{h}"] > 0).astype(int)

print("Labels created")
print("Available target columns:")

target_cols = [col for col in df.columns if col.startswith("return_") or col.startswith("up_")]
print(target_cols)

Labels created
Available target columns:
['return_1', 'up_1', 'return_3', 'up_3', 'return_6', 'up_6', 'return_12', 'up_12']


In [4]:
# ============================================================
# Clean and save processed dataset
# ============================================================

print("Shape before dropna:", df.shape)
print("Missing values before dropna:", df.isna().sum().sum())

df_features = df.dropna().reset_index(drop=True)

print("Shape after dropna:", df_features.shape)
print("Missing values after dropna:", df_features.isna().sum().sum())

df_features.to_csv(OUTPUT_PATH, index=False)

print(f"Saved processed dataset to: {OUTPUT_PATH}")

display(df_features.head())
display(df_features.tail())

Shape before dropna: (723409, 43)
Missing values before dropna: 267
Shape after dropna: (723348, 43)
Missing values after dropna: 0
Saved processed dataset to: ../data/processed\DOGEUSDT_5m_binance_2017_2026_features.csv


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,up_1,future_close_3,return_3,up_3,future_close_6,return_6,up_6,future_close_12,return_12,up_12
0,2019-07-05 13:35:00,0.003815,0.003817,0.003785,0.003809,5726661.0,2019-07-05 13:39:59.999,21762.784020,91,2358281.0,...,1,0.003800,-0.002389,0,0.003822,0.003413,1,0.003863,0.014072,1
1,2019-07-05 13:40:00,0.003812,0.003885,0.003812,0.003840,16701268.0,2019-07-05 13:44:59.999,64066.264668,158,10516524.0,...,0,0.003835,-0.001328,0,0.003836,-0.001198,0,0.003855,0.003776,1
2,2019-07-05 13:45:00,0.003842,0.003880,0.003831,0.003840,21970773.0,2019-07-05 13:49:59.999,84686.627401,137,4464170.0,...,0,0.003831,-0.002344,0,0.003860,0.005287,1,0.003850,0.002787,1
3,2019-07-05 13:50:00,0.003835,0.003860,0.003800,0.003800,6436340.0,2019-07-05 13:54:59.999,24591.400060,90,1357970.0,...,1,0.003822,0.005816,1,0.003919,0.031316,1,0.003867,0.017632,1
4,2019-07-05 13:55:00,0.003820,0.003840,0.003806,0.003835,7182776.0,2019-07-05 13:59:59.999,27471.432703,75,4783851.0,...,0,0.003836,0.000130,1,0.003864,0.007405,1,0.003841,0.001564,1


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,up_1,future_close_3,return_3,up_3,future_close_6,return_6,up_6,future_close_12,return_12,up_12
723343,2026-05-23 09:50:00,0.09950,0.09952,0.09943,0.09946,1679336.0,2026-05-23 09:54:59.999,167050.11271,1127,783613.0,...,0,0.09946,0.000000,0,0.09940,-0.000603,0,0.09952,0.000603,1
723344,2026-05-23 09:55:00,0.09947,0.09948,0.09938,0.09940,1066829.0,2026-05-23 09:59:59.999,106080.10005,780,384552.0,...,1,0.09955,0.001509,1,0.09954,0.001408,1,0.09952,0.001207,1
723345,2026-05-23 10:00:00,0.09941,0.09947,0.09937,0.09942,815385.0,2026-05-23 10:04:59.999,81075.92691,997,222029.0,...,1,0.09949,0.000704,1,0.09963,0.002112,1,0.09948,0.000604,1
723346,2026-05-23 10:05:00,0.09942,0.09948,0.09937,0.09946,3255114.0,2026-05-23 10:09:59.999,323681.19633,1232,2459145.0,...,1,0.09940,-0.000603,0,0.09964,0.001810,1,0.09954,0.000804,1
723347,2026-05-23 10:10:00,0.09945,0.09955,0.09945,0.09955,312988.0,2026-05-23 10:14:59.999,31143.08415,922,176706.0,...,0,0.09954,-0.000100,0,0.09967,0.001205,1,0.09956,0.000100,1
